# Collecting Data - Preprocessing and Data Acquisition Test
This notebook tests the classes for data acquisition, technical indicator calculation, and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path.cwd().parent))

from Collecting_Data.utils import TechnicalIndicators
from Collecting_Data.mt5data import MT5DataLoader
from Collecting_Data.Price import YFinanceDataLoader
from Collecting_Data.preproc_single_inout import SingleInOutPreprocessor
from Collecting_Data.preproc_multi_inout import MultiInOutPreprocessor
from Collecting_Data.preproc_pivot import PivotPreprocessor

## 1. Test Technical Indicators (utils.py)

In [ ]:
# Load sample data
data_path = Path.cwd().parent / "Data" / "GBPUSD_1h.csv"
if data_path.exists():
    df_raw = pd.read_csv(data_path)
    ti = TechnicalIndicators()
    df_indicators = ti.add_all_indicators(df_raw)
    print(f"Indicators added. New shape: {df_indicators.shape}")
    display(df_indicators[['Close', 'EMA_50', 'RSI', 'MACD', 'ATR']].head())
else:
    print("GBPUSD_1h.csv not found, skipping indicator test.")

## 2. Test Data Loaders (mt5data.py & Price.py)

In [ ]:
# Test YFinanceDataLoader
yf_loader = YFinanceDataLoader()
yf_df = yf_loader.download_data(period='1mo', interval='1h')
if yf_df is not None:
    print("YFinance data downloaded successfully.")
    display(yf_df.head())

# Test MT5DataLoader (Requires MT5 Terminal and credentials)
print("\nTesting MT5DataLoader (Mocked initialization test)...")
mt5_loader = MT5DataLoader()
print("MT5DataLoader initialized. (Real connection requires MT5 terminal)")

## 3. Test Preprocessors

In [ ]:
# 3.1 SingleInOutPreprocessor
print("Testing SingleInOutPreprocessor...")
preproc_s = SingleInOutPreprocessor()
df_s = preproc_s.preprocess("GBPUSD_1h.csv")
if df_s is not None:
    print(f"Single InOut shape: {df_s.shape}")
    display(df_s.head())

# 3.2 MultiInOutPreprocessor
print("\nTesting MultiInOutPreprocessor...")
preproc_m = MultiInOutPreprocessor()
X, y = preproc_m.preprocess("GBPUSD_1h.csv", num_input_candles=24, num_output_candles=1)
if X is not None:
    print(f"Multi InOut shapes - X: {X.shape}, y: {y.shape}")

# 3.3 PivotPreprocessor
print("\nTesting PivotPreprocessor...")
preproc_p = PivotPreprocessor()
df_p = preproc_p.preprocess("GBPUSD_1h.csv")
if df_p is not None:
    print(f"Pivot data shape: {df_p.shape}")

## 4. Visualizing Features

In [ ]:
if 'df_indicators' in locals():
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 1, 1)
    plt.plot(df_indicators['Close'].tail(200), label='Close')
    plt.plot(df_indicators['EMA_50'].tail(200), label='EMA 50')
    plt.plot(df_indicators['EMA_200'].tail(200), label='EMA 200')
    plt.title("Price and EMAs")
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(df_indicators['RSI'].tail(200), label='RSI', color='purple')
    plt.axhline(70, linestyle='--', alpha=0.5, color='red')
    plt.axhline(30, linestyle='--', alpha=0.5, color='green')
    plt.title("RSI")
    plt.legend()
    
    plt.tight_layout()
    plt.show()